# 03. 토스광고 CTR 5-Fold LightGBM 정형 베이스라인 & 50:50 WLL 가중치/Prior Shift 비교

이 노트북은 Gate 1에서 확정된 경량 정형 데이터(`train_tabular_diet.parquet`, `test_tabular_diet.parquet`)와 공통 CV 분할(`folds.parquet`) 위에서 **5-Fold LightGBM 베이스라인**을 구축합니다.

### 핵심 실험 목표
1. **공식 지표(`AP 50%` + `50:50 WLL 50%`) 기준선(Baseline) 확립**: OOF 전체·폴드별·OOT(`is_oot_val`) 구간 점수를 분리 추적합니다.
2. **불균형 50:50 WLL 대응 전략 정량 비교**: `scale_pos_weight = N_0 / N_1` 학습 vs 원시 분포 학습 후 `shift_prob_prior(..., target_pos_rate=0.5)` 보정의 `AP`, `WLL`, `Score`를 동일 폴드 위에서 비교합니다.
3. **표준 앙상블 아티팩트 및 초기 제출 검증**: `oof_lgbm_baseline.parquet`, `test_lgbm_baseline.parquet` 저장 규격과 `ID` 순서 무결성을 검증합니다.

In [ ]:
from google.colab import drive
import gc
import importlib.util
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

import os as _os
if not _os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# TODO: Google Drive 내 프로젝트 루트 경로에 맞게 수정하세요.
PROJECT_ROOT = Path('/content/drive/MyDrive/toss_ctr')
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if (PROJECT_ROOT / 'pyproject.toml').exists():
    os.chdir(PROJECT_ROOT)
    %pip install -e . --no-deps --quiet

if importlib.util.find_spec('lightgbm') is None:
    %pip install lightgbm --quiet

from toss_ctr.data import DEFAULT_CONTRACT, load_folds, validate_folds_contract
from toss_ctr.evaluation.metrics import evaluate_predictions
from toss_ctr.models.gbdt_trainer import GBDTOOFTrainer

cfg_path = PROJECT_ROOT / 'configs' / 'gbdt_baseline.yaml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Loaded gbdt_baseline.yaml:', cfg)

### [해석 1] Colab 런타임 연결 및 베이스라인 설정 로드
- Google Drive 마운트 후 `toss_ctr` 패키지를 editable 모드로 연결하고, `lightgbm` 패키지 존재 여부를 먼저 확인한 뒤 누락 시에만 설치합니다.
- `configs/gbdt_baseline.yaml`에서 `seed=42`, `use_50_50_weight=true`, `compare_prior_shift=true`, `test_agg='logit_mean'` 및 범주형 피처 목록(`gender`, `age_group`, `inventory_id`, `day_of_week`, `hour`, `l_feat_14`)을 주입해 실험 재현성을 고정합니다.

In [ ]:
train_path = Path(cfg['paths']['train_path'])
test_path = Path(cfg['paths']['test_path'])
folds_path = Path(cfg['paths']['folds_path'])
preds_dir = Path(cfg['paths']['preds_dir'])
reports_dir = Path(cfg['paths']['reports_dir'])
submission_dir = Path(cfg['paths']['submission_dir'])

train_df = pq.read_table(train_path).to_pandas()
test_df = pq.read_table(test_path).to_pandas()
folds_df = load_folds(folds_path, validate=True, n_rows=len(train_df), n_splits=5)
validate_folds_contract(folds_df, n_rows=len(train_df), n_splits=5)

train_ram_gb = train_df.memory_usage(deep=False).sum() / (1024 ** 3)
test_ram_gb = test_df.memory_usage(deep=False).sum() / (1024 ** 3)
pos_rate = float(train_df[DEFAULT_CONTRACT.target_col].mean())
neg_pos_ratio = float((1.0 - pos_rate) / max(pos_rate, 1e-12))

reserved = set(DEFAULT_CONTRACT.reserved_cols)
base_feature_cols = [c for c in train_df.columns if c not in reserved]

print(f'Train shape: {train_df.shape}, RAM: {train_ram_gb:.2f} GB')
print(f'Test  shape: {test_df.shape}, RAM: {test_ram_gb:.2f} GB')
print(f'CTR (clicked=1 ratio): {pos_rate:.4%} (N_0 / N_1 = {neg_pos_ratio:.2f})')
print(f'Baseline tabular features: {len(base_feature_cols)}개, OOT val rows: {int(folds_df["is_oot_val"].sum()):,}건')

### [해석 2] 경량 Diet 테이블 적재, CV 계약 검증 및 클래스 불균형 확인
- `seq` 컬럼이 분리되고 다운캐스팅된 `train_tabular_diet.parquet` / `test_tabular_diet.parquet`를 적재해 Colab RAM을 안정적으로 유지합니다.
- `validate_folds_contract()`로 `row_id` `0..N-1` 연속성과 5-Fold 분할 무결성을 확인했으며, 극단적 클래스 불균형(`N_0 / N_1`) 때문에 무가중 원시 학습만으로는 `50:50 WLL`의 양성 클래스 손실(`pos_logloss`)이 크게 악화될 수 있음을 수치로 확인합니다.

In [ ]:
# 실험 A: scale_pos_weight = N_0 / N_1 (50:50 균형 가중치 학습)
trainer_bal = GBDTOOFTrainer.from_config(cfg)
res_bal = trainer_bal.fit_predict(
    train_df=train_df,
    folds=folds_df,
    test_df=test_df,
    feature_cols=base_feature_cols,
    categorical_features=cfg.get('categorical_features', []),
)

print('=== [실험 A: 50:50 Balanced Weight (scale_pos_weight = N_0 / N_1)] ===')
print('Overall OOF:', res_bal.overall_metrics.to_dict())
if res_bal.oot_metrics is not None:
    print('OOT Slice  :', res_bal.oot_metrics.to_dict())
display(pd.DataFrame(res_bal.fold_metrics))

### [해석 3] 실험 A (`scale_pos_weight = N_0 / N_1`) 5-Fold 및 OOT 성능 분석
- 각 폴드의 학습 세트 비율에 맞춰 `scale_pos_weight = N_0 / N_1`을 자동 적용하고, 검증 조기 종료(`eval_sample_weight`) 역시 50:50 균형 가중치 기준으로 모니터링해 `WLL`을 직접 최적화했습니다.
- `pos_logloss`와 `neg_logloss`가 한쪽으로 치우치지 않고 균형을 이루는지 확인하며, 5개 폴드 간 `AP`/`WLL`/`Score` 편차와 `is_oot_val` 구간의 시간축 일반화 성능을 함께 점검합니다.

In [ ]:
# 실험 B: 원시 분포 학습 + Log-odds Prior Shift (target_pos_rate=0.5) 비교
comparison_rows = [
    {
        'strategy': 'balanced_weight (scale_pos_weight)',
        **res_bal.overall_metrics.to_dict(),
    }
]

res_shift = None
if cfg.get('compare_prior_shift', True):
    trainer_shift = GBDTOOFTrainer(
        backend=cfg['backend'],
        model_name='lgbm_baseline_prior_shift',
        params=cfg['params'],
        categorical_features=cfg.get('categorical_features', []),
        weighting_mode='prior_shift',
        target_pos_rate=float(cfg.get('target_pos_rate', 0.5)),
        early_stopping_rounds=cfg.get('early_stopping_rounds', 100),
        test_agg=cfg.get('test_agg', 'logit_mean'),
        seed=int(cfg['seed']),
    )
    res_shift = trainer_shift.fit_predict(
        train_df=train_df,
        folds=folds_df,
        test_df=test_df,
        feature_cols=base_feature_cols,
        categorical_features=cfg.get('categorical_features', []),
    )
    if res_shift.unshifted_metrics is not None:
        comparison_rows.append(
            {
                'strategy': 'raw_unweighted (no shift)',
                **res_shift.unshifted_metrics.to_dict(),
            }
        )
    comparison_rows.append(
        {
            'strategy': 'raw_unweighted + prior_shift(0.5)',
            **res_shift.overall_metrics.to_dict(),
        }
    )

comp_df = pd.DataFrame(comparison_rows).sort_values('score', ascending=False).reset_index(drop=True)
display(comp_df)

# 최적 전략 선택 (Combined Score 최대화 기준)
if res_shift is not None and res_shift.overall_metrics.score > res_bal.overall_metrics.score:
    best_result = res_shift
    best_trainer = trainer_shift
    best_result.model_name = cfg['model_name']
    print(f'Selected best baseline: prior_shift (Score={best_result.overall_metrics.score:.6f})')
else:
    best_result = res_bal
    best_trainer = trainer_bal
    print(f'Selected best baseline: balanced_weight (Score={best_result.overall_metrics.score:.6f})')

# 전략별 AP / WLL / Score 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(comp_df))
axes[0].bar(x - 0.18, comp_df['ap'], width=0.35, label='AP', color='#1f77b4')
axes[0].bar(x + 0.18, comp_df['score'], width=0.35, label='Combined Score', color='#2ca02c')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comp_df['strategy'], rotation=15, ha='right', fontsize=9)
axes[0].set_title('AP & Combined Score by Weighting Strategy')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - 0.25, comp_df['wll'], width=0.25, label='50:50 WLL', color='#d62728')
axes[1].bar(x, comp_df['pos_logloss'], width=0.25, label='Pos LogLoss', color='#ff7f0e')
axes[1].bar(x + 0.25, comp_df['neg_logloss'], width=0.25, label='Neg LogLoss', color='#9467bd')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comp_df['strategy'], rotation=15, ha='right', fontsize=9)
axes[1].set_title('WLL Decomposition (Lower is Better)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

gc.collect()

### [해석 4] `scale_pos_weight` vs `raw + shift_prob_prior(0.5)` 비교 및 최적 베이스라인 선정
- **단조 변환의 AP 불변성**: `raw_unweighted (no shift)` 대비 `raw_unweighted + prior_shift(0.5)`는 Log-odds 공간에서 상수 평행이동만 수행하므로 **순위 지표(`AP`)는 유지**되면서도 극단적으로 높았던 `pos_logloss`가 급감해 `WLL`과 `Score`가 크게 개선됩니다.
- **최종 전략 채택**: 동일 5-Fold 위에서 `balanced_weight(scale_pos_weight)`와 `prior_shift(0.5)`의 `Score`를 직접 비교해 우세한 전략을 `lgbm_baseline`으로 확정합니다.

In [ ]:
# 진단 시각화: 폴드 안정성, 클래스별 OOF 확률 분포, Top-20 피처 중요도
fold_df = pd.DataFrame(best_result.fold_metrics)
oof_df = best_result.oof_df
top_fi = best_result.feature_importance.head(20).iloc[::-1]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1) 폴드별 Score / AP / WLL 추이
axes[0].plot(fold_df['fold_id'], fold_df['score'], marker='o', label='Score', color='#2ca02c')
axes[0].plot(fold_df['fold_id'], fold_df['ap'], marker='s', label='AP', color='#1f77b4')
axes[0].plot(fold_df['fold_id'], fold_df['wll'], marker='^', label='WLL', color='#d62728')
axes[0].set_title('5-Fold CV Stability')
axes[0].set_xlabel('Fold ID')
axes[0].set_xticks(fold_df['fold_id'])
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2) clicked=0 vs clicked=1 OOF 확률 분포 (서브샘플 시각화)
rng = np.random.default_rng(42)
pos_probs = oof_df.loc[oof_df['clicked'] == 1, 'pred_prob'].to_numpy()
neg_probs = oof_df.loc[oof_df['clicked'] == 0, 'pred_prob'].to_numpy()
if len(neg_probs) > 200_000:
    neg_probs = rng.choice(neg_probs, size=200_000, replace=False)
if len(pos_probs) > 200_000:
    pos_probs = rng.choice(pos_probs, size=200_000, replace=False)
axes[1].hist(neg_probs, bins=40, density=True, alpha=0.55, label='clicked=0', color='#1f77b4')
axes[1].hist(pos_probs, bins=40, density=True, alpha=0.55, label='clicked=1', color='#ff7f0e')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1.0, label='0.5 threshold')
axes[1].set_title('OOF Predicted Probability Distribution')
axes[1].set_xlabel('pred_prob (50:50 Balanced)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3) Top-20 Gain Feature Importance
axes[2].barh(top_fi['feature'], top_fi['importance_mean'], color='#4c72b0')
axes[2].set_title('Top-20 Feature Importance (Gain)')
axes[2].set_xlabel('Mean Gain across 5 Folds')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

display(best_result.feature_importance.head(20))

### [해석 5] 베이스라인 오차·확률 분포 및 피처 중요도 해석
- **폴드 일관성**: 5개 폴드 간 `AP`와 `WLL` 변동폭이 작으면 시간·클래스 계층화 분할 위에서 모델이 안정적으로 수렴했음을 의미합니다.
- **50:50 확률 분리도**: `clicked=0`과 `clicked=1`의 예측 확률 분포가 `0.5`를 중심으로 좌우 분리되어 있을수록 `WLL` 손실이 낮고 후속 캘리브레이션 여유가 커집니다.
- **핵심 신호 파악**: 상위 Gain 피처에 `inventory_id`, `l_feat_14`, `history_a_*`, `feat_*`가 포진한 양상을 확인하고, 다음 노트북(`04`)에서 이들의 교호작용·OOF 타겟 인코딩·정보영역 결측 패턴을 집중 확장할 근거를 확보합니다.

In [ ]:
# 표준 아티팩트 저장 및 제출 무결성 검증
saved_paths = best_trainer.save_artifacts(best_result, output_dir=preds_dir)
print('Saved artifacts:', saved_paths)

oof_saved = pq.read_table(saved_paths['oof']).to_pandas()
test_saved = pq.read_table(saved_paths['test']).to_pandas()

assert list(oof_saved.columns) == ['row_id', 'fold_id', 'clicked', 'raw_logit', 'pred_prob']
assert list(test_saved.columns) == ['ID', 'raw_logit', 'pred_prob']
assert np.array_equal(oof_saved['row_id'].to_numpy(), np.arange(len(train_df), dtype=np.int64))
assert test_saved['ID'].equals(test_df['ID']), 'Test ID 순서가 원본과 일치하지 않습니다!'
assert not oof_saved.isna().any().any() and not test_saved.isna().any().any()
assert np.isfinite(oof_saved['raw_logit']).all() and np.isfinite(test_saved['raw_logit']).all()
assert ((test_saved['pred_prob'] > 0.0) & (test_saved['pred_prob'] < 1.0)).all()

# 초기 베이스라인 제출 파일 미리보기 저장 (선택)
submission_dir.mkdir(parents=True, exist_ok=True)
sub_path = submission_dir / 'submission_lgbm_baseline.csv'
sub_df = pd.DataFrame({'ID': test_saved['ID'], 'clicked': test_saved['pred_prob']})
sub_df.to_csv(sub_path, index=False)
print(f'Baseline submission verified & written: {sub_path} (rows={len(sub_df):,})')
display(sub_df.head())

### [해석 6] 표준 OOF/Test 아티팩트 규격 및 초기 제출 무결성 확인
- `oof_lgbm_baseline.parquet` (`row_id` 오름차순 `0..N-1`) 및 `test_lgbm_baseline.parquet` (`test_tabular_diet.parquet`의 원본 `ID` 순서 100% 보존)가 결측·무한대 없이 `(0, 1)` 개구간 확률로 저장되었음을 단언(assert) 검증했습니다.
- 저장된 `raw_logit`과 `pred_prob`는 이후 Gate 4(`ensemble-calibrator`)에서 DeepCTR 모델과 Log-odds 블렌딩할 때 그대로 활용됩니다.

### [종합 요약 및 다음 단계 안내]
#### 주요 산출물 및 검증 포인트
- **베이스라인 아티팩트**: `data/processed/predictions/oof_lgbm_baseline.parquet`, `data/processed/predictions/test_lgbm_baseline.parquet`, `data/processed/predictions/lgbm_baseline_metrics.json`, `outputs/submissions/submission_lgbm_baseline.csv` 생성 규격을 확정했습니다.
- **불균형 50:50 WLL 대응**: `scale_pos_weight = N_0 / N_1` 학습과 `raw + shift_prob_prior(..., target_pos_rate=0.5)` 보정의 `AP`/`WLL`/`Score`를 동일 CV 위에서 정량 비교해 최적 베이스라인 기준선을 확보했습니다.
#### 다음 단계 (`04_tabular_fe_and_gbdt.ipynb`)
- 지면(`inventory_id`) × 광고셋(`l_feat_14`) × 맥락(`hour`/`age_group`/`gender`/`day_of_week`) 교호작용, 행 단위 정보영역(`feat_a~e`) 결측 패턴 및 `history_a_*` 비율·차이, `seq_summary` 조인, 누수 제로 `OutOfFoldTargetEncoder`를 적용해 LightGBM·CatBoost·XGBoost 튜닝 모델로 `AP`와 `WLL`을 동시 개선합니다.